# Portfolio Optimization for SFO: from Closed Form to Simulation

A step-by-step implementation of the leading allocation example from A. Meucci, *Risk and Asset Allocation*, Section 6.1, on three real ETFs: **DFUS** (US equity), **DFAI** (international developed), **DFAE** (emerging markets).

## Scope: a minimum viable model
The plan: start from the book's fully analytical solution, then replace one ingredient at a time with its numerical counterpart, requiring at each step that the new answer reproduces the previous one wherever they overlap.

1. **Analytical solution.** Assumptions: horizon prices are (approximately) normal, utility is exponential (CARA), the index of satisfaction is the certainty-equivalent (CE), and risk is monitored through a VaR budget. The optimal allocation is the closed form (6.39).
2. **Convex optimization.** The same problem written as a second-order cone program in `cvxpy` — must match (6.39) to solver tolerance.
3. **VaR → CVaR.** The quantile constraint is replaced by the tail average (expected shortfall). Still closed-form under normality — and, unlike VaR, CVaR remains convex on simulated scenarios, which is what makes step 4 possible.
4. **Exact lognormal market.** Normal invariants actually imply *lognormal* horizon prices, and the portfolio value — a sum of correlated lognormals — has no parametric distribution at all. We switch to Monte Carlo scenarios (Sample Average Approximation) and re-solve, using steps 1–3 as validation anchors.

Fixed throughout: initial wealth $w$ = USD 100M, risk tolerance $\zeta$ = USD 10M, confidence level $c$ = 95%, investment horizon $\tau$ = 1 week.

> **Companion paper.** This is paper 1 of 2: the closed-form path at a one-week horizon, with the lognormal Monte Carlo SAA (step 4) as its validation. Paper 2, [`02_nonparametric_pipeline.ipynb`](02_nonparametric_pipeline.ipynb), drops the normal model and the estimation step altogether: the empirical distribution of the weekly invariants is the model, projected to a **1-year** horizon by a block bootstrap, with cash, scenario probabilities (entropy-pooling stage), the CVaR-constrained optimizer, and an evaluator (unseen scenarios, history resampling, Meucci 2012 liquidity). The roadmap items "1 Year horizon" and "entropy pooling" below are taken up there. The appendix at the end of this paper is the one-week bridge between the two.

## Scope: a practical model

This notebook is deliberately a **minimum viable model**: at every stage of the pipeline we take the simplest defensible choice — three liquid equity ETFs, i.i.d. normal invariants, plain sample moments, trivial projection, CARA utility, no transaction costs. The point of building the full MVM first is methodological: once the end-to-end pipeline is validated (each numerical method checked against a closed form wherever they overlap), every later refinement replaces exactly **one** block while the rest of the machinery — and its checkpoints — stays fixed.

Planned extensions, by pipeline stage:

**Invariants (Ch. 3.1)**
- Add **bonds** (Ch. 3.1.2): the equity recipe does not transfer — a bond's time-to-maturity shrinks, so raw returns are not time-homogeneous; the invariants are changes in yield to maturity at a *fixed* time-to-maturity.
- Add **real estate in Abu Dhabi** (open research question): appraisal-based and illiquid price series violate the i.i.d. invariant construction (smoothing, stale prices); requires its own study before it can enter the model.

**Estimation (Ch. 4)**
- **Shrinkage estimators** of the moments (Ch. 4.4): trade a little bias for a large variance reduction — especially important for the mean vector, which drives the allocation and is the noisiest input.
- **Robust M-estimators** (Ch. 4.5): reduce the influence of outliers on $\hat\mu, \hat\Sigma$.
- **M-estimators combined with shrinkage** (recent literature).

**Projection (Ch. 3.2)**
- **1 Year horizon**
- **Cyclostationary model**: relax "identically distributed across the calendar" — allow periodic (seasonal) structure in the invariant distribution instead of the plain $\tau/\tilde\tau$ scaling.

**Views / beliefs (Part III of the Meucci's)**
- Incorporate subjective beliefs about the future alongside the historical estimates. Two book routes: **Bayesian estimation** (Ch. 7 — prior + data $\to$ posterior distribution of the invariants) and **Black–Litterman** (§9.2 — equilibrium prior blended with views on expected returns). A third, scenario-native route from Meucci's later work is **entropy pooling** ("Fully Flexible Views", 2008): keep the simulated scenarios, reweight their probabilities to satisfy the views with minimal distortion — it plugs directly into the scenario/SAA framework of step 4, which already supports non-uniform scenario weights.

**Utility (Ch 5.4)**
- Move from CARA to the **HARA family**, which nests exponential (used here), power/CRRA and quadratic utility — allowing risk aversion to depend on wealth.

**Optimization**
- Add **transaction costs** (6.14), turning the budget constraint into (6.23) — the cost term $k'|\tilde\alpha - \alpha|$ is convex piecewise-linear, so it slots directly into the convex formulation.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from emfin_capstone.toolbox import price, returns, normal_market, scenario_market
from emfin_capstone import data_types as dt
from datetime import date
from plotly import graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

## Invariants: raw data (Meucci Ch. 3.1)

Market invariants — quantities that repeat i.i.d. across time — are, for equities, the **compounded (log) returns**. Construction choices:

- **Dividend-adjusted** close prices (total return). These funds distribute ≈2%/yr; unadjusted prices would bias the means down.
- **Weekly** frequency, **Wednesday-to-Wednesday** closes, to stay clear of weekend/Monday effects.
- A return is kept only when two consecutive Wednesday closes are exactly 7 calendar days apart. This drops the weeks whose Wednesday is an exchange holiday (2 occurrences in the sample). Weeks with a holiday on some *other* weekday are kept: a Wed→Wed return spans exactly one calendar week regardless of how many trading sessions fall inside it.

We start with equities only, because their invariants are the simplest to construct (bonds and derivatives need the machinery of Ch. 3.1).

In [2]:
asset_symbol_list = ['DFUS', 'DFAI', 'DFAE']
start_date = date(year=2021, month=6, day=15)
end_date = date.today()
price_fetcher_factory = price.PriceFetcherFactory(adjust_prices=True)
price_fetcher = price_fetcher_factory.build_price_fetcher(
    asset_type=dt.AssetType.EQUITY,
    exchange=dt.Exchange.NYSE,
)

prices_dict = {}
weekly_returns_dict = {}
for asset_symbol in asset_symbol_list:
    price_history = price_fetcher.fetch_symbol(symbol=asset_symbol, start=start_date, end=end_date)
    prices_dict[asset_symbol] = pd.Series(
        {observation.date: observation.close for observation in price_history.closes}
    )
    weekly_returns = returns.convert_history_prices_to_weekly_returns(price_history)
    weekly_returns_dict[weekly_returns.symbol] = pd.Series(
        {r.date: r.equity_return for r in weekly_returns.equity_returns}
    )

# Align each column on its own dates: pandas outer-joins the per-symbol indices,
# so assets with differing trading calendars line up automatically.  Weeks missing
# for any asset are dropped from the returns: moments and bootstrap need whole rows.
assets_prices = pd.DataFrame(prices_dict)
assets_returns = pd.DataFrame(weekly_returns_dict).dropna()
print(assets_prices.head())
print(assets_returns.head())

                 DFUS       DFAI       DFAE
2021-06-15  43.718159  26.355700  26.260725
2021-06-16  43.526623  26.173223  25.951050
2021-06-17  43.490955  25.951637  26.066072
2021-06-18  42.953907  25.417221  25.818327
2021-06-21  43.545410  25.734400  25.942202
                DFUS      DFAI      DFAE
2021-06-23  0.005433 -0.020999  0.001914
2021-06-30  0.013287 -0.002051  0.011031
2021-07-07  0.011918  0.003723 -0.018912
2021-07-14  0.000084  0.004117 -0.001377
2021-07-21  0.000105 -0.012643 -0.013875


## Estimation
one-week distribution (Meucci Ch. 4)

Estimation step: the invariant distribution is summarized by the sample mean vector $\hat\mu$ and the sample covariance matrix $\hat\Sigma$ (with Bessel's correction, `ddof=1`).

Working assumption for everything downstream: $X \sim N(\hat\mu, \hat\Sigma)$, i.i.d. across weeks. With ≈266 weekly observations and only 3 assets, plain sample moments are an acceptable starting point. The histogram is a visual sanity check — the fat tails it reveals are deliberately ignored for now (the scenario framework of step 4 can later accommodate any invariant distribution).

Worth noting: **all statistical error of the entire pipeline lives in $\hat\mu$ and $\hat\Sigma$**. Every quantity computed below is deterministic arithmetic on these two estimates.

In [3]:
fig = px.histogram(assets_returns, x='DFUS', title='DFUS weekly log returns')
fig.show()

fig = px.histogram(assets_returns, x='DFAI', title='DFAI weekly log returns')
fig.show()

fig = px.histogram(assets_returns, x='DFAE', title='DFAE weekly log returns')
fig.show()

data = assets_returns.to_numpy()
# 1. Calculate the sample mean vector (for each column)
inv_mean_vector = np.mean(data, axis=0).reshape((3, 1))

# 2. Calculate the sample variance-covariance matrix (ddof=1 for sample/Bessel's correction)
inv_cov_matrix = np.cov(data, rowvar=False, ddof=1)

print("Sample Mean Vector:")
print(inv_mean_vector)

print("\nSample Variance-Covariance Matrix:")
print(inv_cov_matrix)

Sample Mean Vector:
[[0.0022982 ]
 [0.00176365]
 [0.0012075 ]]

Sample Variance-Covariance Matrix:
[[0.00049247 0.00036212 0.00038888]
 [0.00036212 0.00043005 0.00038972]
 [0.00038888 0.00038972 0.00057161]]


## Projection
to the investment horizon (Meucci Ch. 3.2)

Log returns add across time, so i.i.d. invariants project to a horizon $\tau$ by simple scaling (the "square-root rule" for volatility):

$$\mu_\tau = \frac{\tau}{\tilde\tau}\,\hat\mu, \qquad \Sigma_\tau = \frac{\tau}{\tilde\tau}\,\hat\Sigma,$$

where $\tilde\tau$ is the estimation interval. Here the horizon equals the estimation interval (1 week), so the factor is 1 — this cell exists so that the horizon can be changed in a single place.

In [4]:
tau_projected = 1 / 52
tau_in_estimation = 1 / 52
projected_mean_vector = tau_projected / tau_in_estimation * inv_mean_vector
projected_cov_matrix = tau_projected / tau_in_estimation * inv_cov_matrix

## Market distribution (Meucci Ch. 3.3)

Horizon prices follow from the invariants by exact pricing: $P_{T+\tau} = p_T \circ e^{X_\tau}$. With normal invariants the market is therefore **lognormal**, not normal.

Meucci's Section 6.1 example nevertheless assumes $P_{T+\tau} \sim N(\xi, \Phi)$. This is a crude but deliberate approximation: normality of the market is precisely what makes every downstream object (CE, VaR, the optimal allocation itself) closed-form. To make the approximation as faithful as possible, $\xi$ and $\Phi$ are set to the **exact moments of the lognormal**:

$$\xi = p_T \circ e^{\mu_\tau + \frac{1}{2}\operatorname{diag}\Sigma_\tau}, \qquad \Phi = \xi\xi' \circ \left(e^{\Sigma_\tau} - 1\right).$$

Two classic traps hide in these formulas. First, $p \circ e^{\mu}$ alone is the *median* of the lognormal, not its mean — the extra $\frac{1}{2}\operatorname{diag}\Sigma$ is the Jensen correction for the convex map $e^x$ (symmetric log-shocks produce asymmetric dollar moves). Second, the price covariance entangles the means: in log space mean and covariance are clean and separate; the nonlinearity mixes them.

In [5]:
prices_vector = assets_prices.iloc[-1].to_numpy().reshape((3,1))
normal_in_usd_mean_vector = np.log(prices_vector) + projected_mean_vector
normal_in_usd_cov_matrix = projected_cov_matrix
market_mean_vector = np.exp(normal_in_usd_mean_vector + np.diag(normal_in_usd_cov_matrix).reshape((3,1)) / 2) 
market_cov_matrix = market_mean_vector @ market_mean_vector.transpose() * (np.exp(normal_in_usd_cov_matrix) - 1)
print("Sample Mean Vector:")
print(market_mean_vector)

print("\nSample Variance-Covariance Matrix:")
print(market_cov_matrix)

Sample Mean Vector:
[[80.29403915]
 [40.86076881]
 [34.93212649]]

Sample Variance-Covariance Matrix:
[[3.17577423 1.18829065 1.09096576]
 [1.18829065 0.71816222 0.55637594]
 [1.09096576 0.55637594 0.69771003]]


## Utility function and index of satisfaction (Meucci Ch 5.4)

Exponential (CARA) utility over horizon wealth $\psi$:

$$u(\psi) = -e^{-\psi/\zeta}.$$

The risk tolerance $\zeta$ **must carry dollar units** — the argument of an exponential is a pure number. Interpretation: facing a fair $\pm z$ gamble, a CARA investor pays $\approx z^2/2\zeta$ to avoid it. Equivalently $\zeta = w/\lambda$, with $\lambda$ the dimensionless relative risk aversion (here $\lambda = 10$).

Why this utility: for CARA + normal market, the certainty-equivalent is *exactly* mean-variance (6.21), with no approximation:

$$CE(\alpha) = \xi'\alpha - \frac{\alpha'\Phi\alpha}{2\zeta}.$$

Note that CE is denominated in **horizon** dollars: it is the sure amount *at the horizon* equivalent to holding the risky portfolio. The definition $u(CE) = E[u(\Psi_\alpha)]$ never references time $T$, and there is no discounting anywhere in this model.

In [6]:
risk_tolerance = 10e6
def exp_utility_function(risk_tolerance: float, wealth_value: float) -> float:
    return - np.exp( - 1 / risk_tolerance * wealth_value)
wealth_value_list = np.arange(0, 100) * 1e6
utility_function = [exp_utility_function(risk_tolerance, x) for x in wealth_value_list]
fig = go.Figure()
fig.add_trace(go.Scatter(x=wealth_value_list, y=utility_function, mode='lines'))
fig.show()

## Portfolio optimization

The allocation problem (6.34): maximize CE subject to the full-investment budget $p_T'\alpha = w$ and a budget-at-risk constraint (risk measure $\le \gamma\,w$).

In [7]:
wealth = 100e6 
confidence_level = 0.95

### Analytical solution (6.39)

The closed-form maximizer of CE under the budget constraint alone:

$$\alpha^* = \zeta\,\Phi^{-1}\xi + \frac{w - \zeta\,p_T'\Phi^{-1}\xi}{p_T'\Phi^{-1}p_T}\,\Phi^{-1}p_T$$

— a *speculative* term (notably independent of wealth: a CARA signature) plus a *budget-absorbing* term. The risk constraint is not imposed here; we compute VaR (6.22) ex post and compare it with the risk budget. VaR is quoted in the **loss convention**: a positive number is a potential loss, $P(\text{weekly loss} > \mathrm{VaR}) = 5\%$.

In [8]:
solution = normal_market.analytical_solution(
    market_mean_vector=market_mean_vector,
    market_cov_matrix=market_cov_matrix,
    prices_vector=prices_vector,
    wealth=wealth,
    risk_tolerance=risk_tolerance,
    confidence_level=confidence_level,
)

Allocations: [[ 887074.42222705]
 [1323631.77788782]
 [-717416.53520107]]
Certain equivalent: [[1.00027436e+08]]
VaR: [[3223819.01052093]]


### Convex optimization — VaR constraint (SOCP in cvxpy)

The same problem solved numerically, with three practical lessons baked into the formulation:

1. **Nondimensionalize.** Solvers want all quantities $O(1)$. Dollar-scale data (shares $\sim 10^5$, wealth $\sim 10^8$, quadratic coefficient $\sim 10^{-7}$) spans 14 orders of magnitude and breaks interior-point linear algebra. We therefore optimize portfolio *weights* $h_i = p_i\alpha_i/w$ with a unit budget; the natural data become gross returns $g = \xi / p_T$ and the per-dollar covariance $\Psi = \Phi / p_Tp_T'$.
2. **DCP rules.** `cvxpy` certifies convexity syntactically, so $\sqrt{h'\Psi h}$ must be written as `cp.norm(L.T @ h)` with $L$ the Cholesky factor of $\Psi$ — this *is* the second-order cone representation.
3. **Risk in the loss convention.** With $k_{VaR} = \sqrt{2}\,\operatorname{erf}^{-1}(2c-1) \approx 1.645$, VaR per unit of wealth reads $(1 - g'h) + k_{VaR}\,\sigma_{rel}$ and is constrained $\le \gamma_{VaR}$.

We set $\gamma_{VaR}$ to the VaR of the analytical allocation (6.39) — `value_at_risk / wealth`. The constraint therefore just binds at the closed-form optimum, so this program reproduces (6.39) to solver tolerance ($\sim 10^{-5}$ relative): a check that the SOCP formulation matches the analytical result. The printed budget dual is a shadow price — one extra dollar of budget buys that many dollars of certainty-equivalent.

In [9]:
gamma_var = (solution.value_at_risk / wealth).item()  # match the analytical VaR budget

normal_market.optimization_with_var_constrain(
    market_mean_vector=market_mean_vector,
    market_cov_matrix=market_cov_matrix,
    prices_vector=prices_vector,
    wealth=wealth,
    risk_tolerance=risk_tolerance,
    confidence_level=confidence_level,
    gamma=gamma_var,
)

Allocations: [[ 886709.19660459]
 [1323740.44120538]
 [-716704.96329626]]
CE: 100,027,436
budget shadow price (CE gained per extra dollar of budget): 0.9982


array([[ 886709.19660459],
       [1323740.44120538],
       [-716704.96329626]])

### Convex optimization — CVaR constraint

The same SOCP, now with the **CVaR** (normal expected-shortfall) risk measure: $k_{CVaR} = \dfrac{e^{-[\operatorname{erf}^{-1}(2c-1)]^2}}{(1-c)\sqrt{2\pi}} \approx 2.063$, so CVaR per unit of wealth reads $(1 - g'h) + k_{CVaR}\,\sigma_{rel}$ and is constrained $\le \gamma_{CVaR} = 4.1\%$ of wealth.

At equal confidence CVaR $>$ VaR always. Because the CVaR budget binds, the optimum legitimately differs a little from (6.39) — the closed form knows nothing about the risk constraint.

In [10]:
gamma_cvar = 0.041

alpha_cvar_normal = normal_market.optimization_with_cvar_constrain(
    market_mean_vector=market_mean_vector,
    market_cov_matrix=market_cov_matrix,
    prices_vector=prices_vector,
    wealth=wealth,
    risk_tolerance=risk_tolerance,
    confidence_level=confidence_level,
    gamma=gamma_cvar,
)

Allocations: [[7.81689917e+05]
 [9.16979837e+05]
 [5.94476346e-01]]
CE: 100,021,167
budget shadow price (CE gained per extra dollar of budget): 0.9981


# Scenario Engine

From this point on, the market distribution enters the optimization only as a
**matrix of sampled scenarios** (with uniform probabilities, so far implicit).
Three things make this section more than a re-implementation of the closed-form
solution:

1. **Estimation is fully decoupled from optimization.** The optimizer consumes
   scenarios and never learns where they came from. Today they are Monte Carlo
   draws from the lognormal market implied by normal invariants; tomorrow the
   *same, unchanged* optimization code accepts Student-t invariants, bootstrapped
   historical weeks, entropy-pooling-reweighted scenario sets, or handcrafted
   stress cases — genuinely any distribution model that can produce samples.
   This makes the model very flexible: every upstream refinement in the
   roadmap (estimation, projection, views) plugs in without touching the
   optimizer.

2. **Constrained optimization on an empirical distribution — via
   Rockafellar–Uryasev.** On scenarios, quantile-based risk measures (VaR) are
   non-convex step functions of the allocation. The R–U minimization formula
   represents CVaR through an auxiliary threshold variable and positive parts,
   keeping the problem convex — the single result that makes the whole
   *constrained* optimization step possible on empirical distributions
   (references in `scenario_market`).

3. **The utility function is still hardwired.** The objective is the empirical
   CARA certainty-equivalent (`log_sum_exp`), so exponential utility remains
   baked into the convex formulation. Generalizing the objective to a pluggable
   utility family (HARA) is the next task in building the scenario engine; the
   requirement on any candidate utility is concavity plus DCP-expressibility.

In [11]:
# Sample from equity invariants to approximate log-normal market distribution
N_SCENARIOS = 400000
SEED = 42


def sample_normal(seed: int) -> np.ndarray:
    return scenario_market.sample_market_scenarios(
        prices_vector=prices_vector,
        projected_mean_vector=projected_mean_vector,
        projected_cov_matrix=projected_cov_matrix,
        n_scenarios=N_SCENARIOS,
        seed=seed,
    )


market_samples = sample_normal(SEED)

# bin in numpy and plot bars: go.Histogram would embed all raw samples
# (3 x 400k floats, several MB) into the notebook JSON on every save.
fig = make_subplots(rows=3, cols=1, subplot_titles=asset_symbol_list)
for row_index in range(len(asset_symbol_list)):
    counts, bin_edges = np.histogram(market_samples[:, row_index], bins=200)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    fig.add_trace(go.Bar(x=bin_centers, y=counts, marker_line_width=0), row=row_index + 1, col=1)
fig.show()

In [12]:
gamma_cvar = 0.041

alpha_scenario = scenario_market.optimization_with_cvar_constrain(
    market_samples=market_samples,
    prices_vector=prices_vector,
    wealth=wealth,
    risk_tolerance=risk_tolerance,
    confidence_level=confidence_level,
    gamma=gamma_cvar,
)

status: optimal
Allocations: [ 704951. 1067693.       0.]
CE (in-solver, optimistically biased): 100,061,130
VaR proxy t*: 3,169,224


In [13]:
evaluation = scenario_market.evaluate_allocation(
    allocations=alpha_scenario,
    market_samples=market_samples,
    sample_scenarios=lambda seed: (sample_normal(seed), scenario_market.uniform_prior(N_SCENARIOS)),
    wealth=wealth,
    risk_tolerance=risk_tolerance,
    confidence_level=confidence_level,
    gamma=gamma_cvar,
    out_of_sample_seed=SEED + 1,
)

full set     : CE 100,021,109   CVaR 3,924,355   E[value] 100,230,134   (budget 4,100,000)
out-of-sample: CE 100,017,959   CVaR 3,917,337   E[value] 100,226,396   (budget 4,100,000)


## Comparing the three approaches: one judge, three defendants

Each optimizer reports a CE under its own model — (6.39) and the SOCP under the
normal approximation $N(\xi, \Phi)$, the SAA under its in-sample scenarios (with
the in-sample optimism bias). Those self-reports are mutually incomparable, so we
grade all three allocations under a **single judge**: the exact lognormal market,
represented by fresh-seed scenario sets that none of the optimizers ever saw.

Protocol: the same judge seeds for every allocation (paired comparison — common
draws cancel, so the CE *differences* are far more precise than either CE alone),
replicated over several seeds so every number carries an error bar. Differences
smaller than their spread across seeds are indistinguishable at this sample size.

Expected reading: allocations differ by millions of dollars while CE differs by
tens of thousands — the mean-variance valley is flat, so portfolios must be
compared by consequences (CE, CVaR), never by coordinates ($\alpha$).

In [14]:
allocations_dict = {
    "analytical (6.39)": solution.allocations,
    "normal-model CVaR": alpha_cvar_normal,
    "scenario SAA CVaR": alpha_scenario,
}

N_JUDGE_SEEDS = 10
JUDGE_SEED_BASE = 1000

results_dict = {}
for judge_seed in range(JUDGE_SEED_BASE, JUDGE_SEED_BASE + N_JUDGE_SEEDS):
    judge_samples = scenario_market.sample_market_scenarios(
        prices_vector=prices_vector,
        projected_mean_vector=projected_mean_vector,
        projected_cov_matrix=projected_cov_matrix,
        n_scenarios=N_SCENARIOS,
        seed=judge_seed,
    )
    for name, allocations in allocations_dict.items():
        judge_wealth = judge_samples @ allocations
        ce = scenario_market.empirical_certainty_equivalent(judge_wealth, risk_tolerance)
        cvar = scenario_market.empirical_cvar(wealth - judge_wealth, confidence_level)
        results_dict.setdefault(name, []).append((ce, cvar))

best_mean_ce = max(np.mean([ce for ce, _ in results]) for results in results_dict.values())
for name, results in results_dict.items():
    ce_array = np.array([ce for ce, _ in results])
    cvar_array = np.array([cvar for _, cvar in results])
    feasibility = "OK" if cvar_array.mean() <= gamma_cvar * wealth else "BREACH"
    print(f"{name:20s} CE {ce_array.mean():,.0f} (+-{ce_array.std():,.0f})   "
          f"dCE vs best {ce_array.mean() - best_mean_ce:+,.0f}   "
          f"CVaR {cvar_array.mean():,.0f} (+-{cvar_array.std():,.0f})   {feasibility}  "
          f"(budget {gamma_cvar * wealth:,.0f})")

analytical (6.39)    CE 100,027,707 (+-1,454)   dCE vs best +0   CVaR 4,033,019 (+-7,349)   OK  (budget 4,100,000)
normal-model CVaR    CE 100,021,452 (+-1,525)   dCE vs best -6,255   CVaR 3,947,269 (+-7,089)   OK  (budget 4,100,000)
scenario SAA CVaR    CE 100,021,061 (+-1,503)   dCE vs best -6,645   CVaR 3,920,767 (+-6,972)   OK  (budget 4,100,000)


# Appendix — bridge to paper 2: the block-bootstrap market at one week

The first step from the closed-form path to the non-parametric pipeline, kept here because every statement in it is a **one-week** statement. Same horizon as above; two changes to the market: a riskless **cash** column (constant at `deposit_rate`, compounded to the horizon) and, as the scenario model, the **stationary block bootstrap of the historical weeks** — with the lognormal Monte Carlo kept alongside as the benchmark. At one week the block length is inert (only a path's first week is drawn), so these scenarios are simply the ~270 historical weeks, resampled.

The bootstrap itself, the block-length choice and its caveats (block vs i.i.d. vs filtered historical simulation), the money-market model and its limitations are developed in paper 2, [`02_nonparametric_pipeline.ipynb`](02_nonparametric_pipeline.ipynb), where the horizon is one year.

In [15]:
# Scenarios: stationary block bootstrap of the historical weekly invariants (whole
# weeks resampled, summed over the horizon); the lognormal Monte Carlo of the normal
# model is kept alongside as the benchmark.  Both carry the same cash column.
N_SCENARIOS = 400000
SEED = 42

weekly_log_returns = assets_returns.to_numpy()
horizon_weeks = round(tau_projected / tau_in_estimation)
# rule of thumb n^(1/3) (6 weeks on ~270): weekly r shows no autocorrelation, r^2 does
# (Politis-White gives ~2-3 on r, ~11-14 on r^2 for DFUS/DFAI, and is unstable).
# Inert at horizon_weeks = 1: only a path's first week is ever drawn.
mean_block_length = round(len(weekly_log_returns) ** (1 / 3))

cash_price = 1.0
deposit_rate = 0.03
cash_horizon_value = cash_price * (1 + deposit_rate) ** tau_projected
print(cash_horizon_value)
prices_with_cash = np.append(prices_vector.ravel(), cash_price)


def sample_bootstrap_with_cash(seed: int) -> np.ndarray:
    market_samples = scenario_market.sample_block_bootstrap_scenarios(
        weekly_log_returns=weekly_log_returns,
        prices_vector=prices_vector,
        horizon_weeks=horizon_weeks,
        n_scenarios=N_SCENARIOS,
        mean_block_length=mean_block_length,
        seed=seed,
    )
    return scenario_market.append_cash_column(market_samples, cash_horizon_value)


def sample_normal_with_cash(seed: int) -> np.ndarray:
    market_samples = scenario_market.sample_market_scenarios(
        prices_vector=prices_vector,
        projected_mean_vector=projected_mean_vector,
        projected_cov_matrix=projected_cov_matrix,
        n_scenarios=N_SCENARIOS,
        seed=seed,
    )
    return scenario_market.append_cash_column(market_samples, cash_horizon_value)


market_samples_with_cash = sample_bootstrap_with_cash(SEED)
normal_samples_with_cash = sample_normal_with_cash(SEED)

# bin in numpy and plot bars: go.Histogram would embed all raw samples
# (3 x 200k floats, several MB) into the notebook JSON on every save.
# At horizon_weeks = 1 the bootstrap has only len(weekly_log_returns) distinct values.
fig = make_subplots(rows=4, cols=1, subplot_titles=asset_symbol_list + ['Cash'])
for row_index, asset_symbol in enumerate(asset_symbol_list):
    counts, bin_edges = np.histogram(market_samples_with_cash[:, row_index], bins=200)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    fig.add_trace(go.Bar(x=bin_centers, y=counts, marker_line_width=0), row=row_index + 1, col=1)
    
counts, bin_edges = np.histogram(market_samples_with_cash[:, 3], bins=200)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
fig.add_trace(go.Bar(x=bin_centers, y=counts, marker_line_width=0), row=4, col=1)

fig.show()

1.0005686000964287


In [16]:
gamma_cvar = 0.041

alpha_scenario = scenario_market.optimization_with_cvar_constrain(
    market_samples=market_samples_with_cash,
    prices_vector=prices_with_cash,
    wealth=wealth,
    risk_tolerance=risk_tolerance,
    confidence_level=confidence_level,
    gamma=gamma_cvar,
)

# benchmark: the same SAA on the lognormal Monte Carlo scenarios
alpha_scenario_normal_mc = scenario_market.optimization_with_cvar_constrain(
    market_samples=normal_samples_with_cash,
    prices_vector=prices_with_cash,
    wealth=wealth,
    risk_tolerance=risk_tolerance,
    confidence_level=confidence_level,
    gamma=gamma_cvar,
)

status: optimal
Allocations: [  300950.   305314.        0. 63446237.]
CE (in-solver, optimistically biased): 100,086,235
VaR proxy t*: 2,239,119


status: optimal
Allocations: [  489825.   258903.        0. 50211834.]
CE (in-solver, optimistically biased): 100,113,183
VaR proxy t*: 2,255,516


In [17]:
evaluation = scenario_market.evaluate_allocation(
    allocations=alpha_scenario,
    market_samples=market_samples_with_cash,
    sample_scenarios=lambda seed: (sample_bootstrap_with_cash(seed), scenario_market.uniform_prior(N_SCENARIOS)),
    wealth=wealth,
    risk_tolerance=risk_tolerance,
    confidence_level=confidence_level,
    gamma=gamma_cvar,
    out_of_sample_seed=SEED + 1,
)

full set     : CE 100,093,234   CVaR 1,787,450   E[value] 100,121,874   (budget 4,100,000)
out-of-sample: CE 100,092,795   CVaR 1,794,593   E[value] 100,121,579   (budget 4,100,000)


## Comparing the approaches: one judge at a time, four defendants

Each optimizer reports a CE under its own model — (6.39) and the SOCP under the
normal approximation $N(\xi, \Phi)$, the two SAAs under their in-sample scenarios
(with the in-sample optimism bias). Those self-reports are mutually incomparable,
so we grade all allocations under a **single judge** at a time, represented by
fresh-seed scenario sets that none of the optimizers ever saw: the block-bootstrap
market (the scenario model in use), and the exact lognormal market as the
benchmark. The closed-form allocations hold no cash; they are graded with a zero
cash position.

Caveat at a one-week horizon: every bootstrap scenario set — in-sample or fresh
seed — is a resample of the same ~270 historical weeks, so the bootstrap judge
measures only Monte Carlo noise, not estimation error. It is the empirical
distribution, not out-of-sample data.

Protocol: the same judge seeds for every allocation (paired comparison — common
draws cancel, so the CE *differences* are far more precise than either CE alone),
replicated over several seeds so every number carries an error bar. Differences
smaller than their spread across seeds are indistinguishable at this sample size.

Expected reading: allocations differ by millions of dollars while CE differs by
tens of thousands — the mean-variance valley is flat, so portfolios must be
compared by consequences (CE, CVaR), never by coordinates ($\alpha$).

In [18]:
no_cash = np.zeros((1, 1))
allocations_dict = {
    "analytical (6.39)": np.vstack([solution.allocations, no_cash]),
    "normal-model CVaR": np.vstack([alpha_cvar_normal, no_cash]),
    "SAA CVaR normal MC": alpha_scenario_normal_mc,
    "SAA CVaR bootstrap": alpha_scenario,
}
judges_dict = {
    "block bootstrap": sample_bootstrap_with_cash,
    "lognormal MC": sample_normal_with_cash,
}

N_JUDGE_SEEDS = 10
JUDGE_SEED_BASE = 1000

for judge_name, sample_judge in judges_dict.items():
    results_dict = {}
    for judge_seed in range(JUDGE_SEED_BASE, JUDGE_SEED_BASE + N_JUDGE_SEEDS):
        judge_samples = sample_judge(judge_seed)
        for name, allocations in allocations_dict.items():
            judge_wealth = judge_samples @ allocations
            ce = scenario_market.empirical_certainty_equivalent(judge_wealth, risk_tolerance)
            cvar = scenario_market.empirical_cvar(wealth - judge_wealth, confidence_level)
            results_dict.setdefault(name, []).append((ce, cvar))

    print(f"judge: {judge_name}")
    best_mean_ce = max(np.mean([ce for ce, _ in results]) for results in results_dict.values())
    for name, results in results_dict.items():
        ce_array = np.array([ce for ce, _ in results])
        cvar_array = np.array([cvar for _, cvar in results])
        feasibility = "OK" if cvar_array.mean() <= gamma_cvar * wealth else "BREACH"
        print(f"{name:20s} CE {ce_array.mean():,.0f} (+-{ce_array.std():,.0f})   "
              f"dCE vs best {ce_array.mean() - best_mean_ce:+,.0f}   "
              f"CVaR {cvar_array.mean():,.0f} (+-{cvar_array.std():,.0f})   {feasibility}  "
              f"(budget {gamma_cvar * wealth:,.0f})")

judge: block bootstrap
analytical (6.39)    CE 100,020,038 (+-3,925)   dCE vs best -73,981   CVaR 5,151,916 (+-9,281)   BREACH  (budget 4,100,000)
normal-model CVaR    CE 100,014,807 (+-3,591)   dCE vs best -79,212   CVaR 4,964,445 (+-9,350)   BREACH  (budget 4,100,000)
SAA CVaR normal MC   CE 100,094,019 (+-1,817)   dCE vs best +0   CVaR 2,501,745 (+-4,486)   OK  (budget 4,100,000)
SAA CVaR bootstrap   CE 100,093,900 (+-1,323)   dCE vs best -120   CVaR 1,784,515 (+-3,396)   OK  (budget 4,100,000)


judge: lognormal MC
analytical (6.39)    CE 100,027,707 (+-1,454)   dCE vs best -65,942   CVaR 4,033,019 (+-7,349)   OK  (budget 4,100,000)
normal-model CVaR    CE 100,021,452 (+-1,525)   dCE vs best -72,197   CVaR 3,947,269 (+-7,089)   OK  (budget 4,100,000)
SAA CVaR normal MC   CE 100,093,649 (+-806)   dCE vs best +0   CVaR 1,987,865 (+-3,790)   OK  (budget 4,100,000)
SAA CVaR bootstrap   CE 100,093,326 (+-571)   dCE vs best -323   CVaR 1,413,128 (+-2,621)   OK  (budget 4,100,000)
